In [ ]:
# Single-Seed Run for Baseline Comparison. Run this 3 times with different SEEDS values. Then use the aggregation script to combine results

!pip -q install -U transformers datasets accelerate evaluate jiwer librosa soundfile torchaudio matplotlib

import os
import json
import hashlib
from datetime import datetime
import random
import gc
import warnings

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, IterableDataset
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import WhisperForConditionalGeneration, WhisperProcessor, get_linear_schedule_with_warmup, logging as hf_logging
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

# Configs:
# Update this with the correct folder if you wanna run cache:
BASE_CACHE = os.environ.get("BASE_CACHE", os.path.join(os.getcwd()))
os.makedirs(BASE_CACHE, exist_ok=True)
os.environ["HF_HOME"] = BASE_CACHE
os.environ["HF_DATASETS_CACHE"] = os.path.join(BASE_CACHE, "datasets")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(BASE_CACHE, "transformers")
os.environ["HF_MODULES_CACHE"] = os.path.join(BASE_CACHE, "modules")
os.environ["TMPDIR"] = os.path.join(BASE_CACHE, "tmp")
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

# Single seed configuration
# Change this value to run different seeds: 3, 17, 24, 35, or 50
TARGET_SEED = 3

RUN_DIR = os.path.join(BASE_CACHE, "runs", f"seed_{TARGET_SEED}")
os.makedirs(RUN_DIR, exist_ok=True)
print(f"RUN_DIR: {RUN_DIR}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cpu = torch.device("cpu")
print("Device:", device)

# Update these hyperparameters for experimentation
BATCH_SIZE = 4
E22_A_PRETRAIN_STEPS = 100
E22_A_FISHER_N = 800
MINIKFAC_FISHER_N = 800
MINIKFAC_MAX_ROWS = 100
MINIKFAC_LAYER_STRIDE = 1
MINIKFAC_ONLY_ENCODER = False
MINIKFAC_MAX_FEATURES = 2048
MINIKFAC_ACCUM_ON_CPU = True
MINIKFAC_STRICT_LAYERWISE = True
MINIKFAC_LAYER_SCALE_MIN = 1.0
MINIKFAC_LAYER_SCALE_MAX = 5.0
MINIKFAC_TRACE_EXPONENT = 0.5
E22_EVAL_N = 200
TRAIN_STEPS = 120
EVAL_INTERVAL = 20
GRAD_CLIP_NORM = 1.0
EARLY_STOP_PATIENCE_EVALS = 6
EARLY_STOP_MIN_DELTA = 0.0
STAGEB_LR = 3e-6
GRAD_ACCUM_STEPS = 2
STAGEB_WARMUP_RATIO = 0.1
GRAD_DIAG_INTERVAL = 5
EWC_RAMP_STEPS = 50
EVAL_MAX_BATCHES = 55
MINIKFAC_LAMBDA_SCALE = 48.0
EWC_LAMBDA_SCALE = 10000.0

STAGE_A_SOURCE = "ted"
SPGI_CONFIG = "S"
STAGE_A_SHUFFLE_BUFFER = 2000
STAGE_A_SOURCE_LABEL = None
CURRENT_RUN_SEED = 0

USE_STAGEA_CACHE = True
STAGEA_CACHE_DIR = os.path.join(BASE_CACHE, "stageA_cache")
os.makedirs(STAGEA_CACHE_DIR, exist_ok=True)

LAMBDA_BASE = 5000.0
LAMBDA_ENC = 10000.0
LAMBDA_DEC = 2500.0
TIE_EPS = 0.003

wer_metric = evaluate.load("wer")
normalizer = BasicTextNormalizer()

# Normalize text using Whisper's BasicTextNormalizer
# input: s (string to normalize)
# process: Apply Whisper text normalization (lowercasing, punctuation removal,...)
# output: Normalized string
def norm(s: str) -> str:
    return normalizer(s)

# Set random seed for reproducibility across all libraries
# input: seed (integer)
# process: Set seed for randomness
# output: None
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Memory cleanup (garbage collection + GPU cache clear); if not used, memory might blow up and process might die.
# input: None
# process: gc.collect(), torch.cuda.empty_cache() if CUDA available
# output: None
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

model_name = "openai/whisper-small"
GEN_KW = dict(language="en", task="transcribe", max_new_tokens=128, num_beams=1)
MAX_LABEL_LEN = 448
SR = 16000

# Convert audio object to 1D float array and sample rate
# input: audio_obj (HF audio dict or object with get_all_samples method)
# process: Extract waveform and sample rate, handle stereo-mono conversion from datasets
# output: (waveform: np.array, sample_rate: int)
def audio_to_1d_float(audio_obj):
    if hasattr(audio_obj, "get_all_samples"):
        s = audio_obj.get_all_samples()
        wav = s.data
        sr = int(s.sample_rate)
        if wav.ndim == 2:
            wav = wav[0]
        return wav.to(cpu).float().numpy(), sr
    wav = audio_obj["array"]
    sr = int(audio_obj["sampling_rate"])
    if hasattr(wav, "ndim") and wav.ndim == 2:
        wav = wav[0]
    return wav, sr

# Streaming dataset wrapper for audio-text pairs
# input: stream (iterable), audio_key, text_key, tokenizer, feature_extractor
# process: Iterates through stream, extracts audio/text, handles missing fields, resamples audio to 16kHz, extracts Whisper features & tokenizes text
# output: Iterator yielding dicts {input_features, labels, text}
class StreamSegments(IterableDataset):
    def __init__(self, stream, audio_key, text_key, tokenizer, feature_extractor):
        self.stream = stream
        self.audio_key = audio_key
        self.text_key = text_key
        self.tokenizer = tokenizer
        self.feature_extractor = feature_extractor

# Streaming dataset wrapper for audio-text pairs
# input: None (uses self.stream, self.audio_key, self.text_key, etc.)
# process: Iterates stream, resamples audio to 16kHz, tokenizes text, yields batch
# output: Generator yielding {input_features, labels, text}
    def __iter__(self):
        for ex in self.stream:
            if isinstance(self.text_key, (list, tuple)):
                t = None
                for k in self.text_key:
                    if k in ex and ex[k] is not None:
                        t = ex[k]
                        break
                if t is None:
                    continue
            else:
                if self.text_key not in ex or ex[self.text_key] is None:
                    continue
                t = ex[self.text_key]
            wav, sr = audio_to_1d_float(ex[self.audio_key])
            if sr != SR:
                import torchaudio
                wav = torchaudio.functional.resample(
                    torch.from_numpy(wav), orig_freq=sr, new_freq=SR
                ).numpy()
                sr = SR

            feats = self.feature_extractor(
                wav,
                sampling_rate=sr,
                return_tensors="pt",
            ).input_features.squeeze(0).to(cpu)
            labels = self.tokenizer(
                t,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LABEL_LEN,
            ).input_ids.squeeze(0).to(cpu)
            yield {"input_features": feats, "labels": labels, "text": t, "_tokenizer": self.tokenizer}

# Create DataLoader from streaming dataset
# input: stream (dataset), audio_key, text_key, tokenizer, feature_extractor, batch_size
# process: From StreamSegments, defines collate_fn for padding, then creates DataLoader
# output: DataLoader that yields {input_features, labels, texts}
def make_stream_loader(stream, audio_key, text_key, tokenizer, feature_extractor, batch_size=2):
    def collate_fn(batch):
        input_features = torch.stack([b["input_features"] for b in batch])
        labels = torch.nn.utils.rnn.pad_sequence(
            [b["labels"] for b in batch],
            batch_first=True,
            padding_value=tokenizer.pad_token_id,
        )
        labels = labels.masked_fill(labels == tokenizer.pad_token_id, -100)
        return {
            "input_features": input_features,
            "labels": labels,
            "texts": [b["text"] for b in batch],
        }

    return DataLoader(
        StreamSegments(stream, audio_key, text_key, tokenizer, feature_extractor),
        batch_size=batch_size,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
        collate_fn=collate_fn,
    )

# Hash string to integer bucket for deterministic splitting
# input: s (string to hash), mod (modulo for bucketing, default 100)
# process: MD5 hash string, convert to int, mod operation
# output: Integer in range [0, mod)
def _h_bucket(s: str, mod: int = 100) -> int:
    return int(hashlib.md5(s.encode("utf-8")).hexdigest(), 16) % mod

# Split Earnings22 sample to train/val/test split
# input: file_id (unique)
# process: Hash file_id if [0,80) = train, [80,90) =val, [90,100) =test
# output: 'train', 'val', or 'test'
def e22_global_split(file_id: str) -> str:
    b = _h_bucket(file_id, 100)
    if b < 80:
        return "train"
    if b < 90:
        return "val"
    return "test"

# Assign Earnings22 sample to Task A or Task B
# input: file_id (unique identifier)
# process: Hash file_id with '::task' suffix: if <50 then A, else >=50 then B
# output: 'A' or 'B'
def e22_task_split(file_id: str) -> str:
    return "A" if _h_bucket(file_id + "::task", 100) < 50 else "B"

# Stream Earnings22 dataset filtered by split and task.
# input: split_name ('train'/'val'/'test'), task_name ('A'/'B'/None for both), shuffle (bool), shuffle_buffer (int), take_n (max samples or None)
# process: Loads Earnings22 streaming dataset, applies e22_global_split/e22_task_split, filters, shuffling, limiting
# output: Iterator samples from filtered Earnings22 subset
class Earnings22TaskStream(IterableDataset):
    def __init__(self, split_name, task_name=None, shuffle=False, shuffle_buffer=5000, take_n=None):
        self.split_name = split_name
        self.task_name = task_name
        self.shuffle = shuffle
        self.shuffle_buffer = shuffle_buffer
        self.take_n = take_n

# Iterate through Earnings22 dataset with split/task filtering
# input: None (uses self.stream, self.split_name, self.task_name)
# process: Streams Earnings22 samples, applies split/task filter via hashing
# output: Generator {audio, text} for specified split/task
    def __iter__(self):
        stream = load_dataset("distil-whisper/earnings22", "chunked", split="test", streaming=True)
        if self.shuffle:
            stream = stream.shuffle(seed=CURRENT_RUN_SEED, buffer_size=self.shuffle_buffer)
        seen = 0
        for ex in stream:
            fid = str(ex.get("file_id", ""))
            if not fid:
                continue
            if e22_global_split(fid) != self.split_name:
                continue
            if self.task_name is not None and e22_task_split(fid) != self.task_name:
                continue
            yield ex
            seen += 1
            if self.take_n is not None and seen >= self.take_n:
                break

# Create DataLoader for Earnings22 subset
# input: split_name, task_name (optional), batch_size, shuffle, take_n, tokenizer, feature_extractor
# process: Create Earnings22TaskStream, wraps in make_stream_loader
# output: DataLoader for batched Earnings22 samples
def make_e22_loader(split_name, task_name=None, batch_size=2, shuffle=False, take_n=None, tokenizer=None, feature_extractor=None):
    ds = Earnings22TaskStream(
        split_name=split_name,
        task_name=task_name,
        shuffle=shuffle,
        take_n=take_n,
    )
    return make_stream_loader(ds, audio_key="audio", text_key="transcription", tokenizer=tokenizer, feature_extractor=feature_extractor, batch_size=batch_size)

# Create Stage-A pretraining dataset stream (TEDLIUM).
# input: shuffle (bool), take_n (max samples or None)
# process: Tries to load TEDLIUM. Optional shuffling and limiting.
# output: (stream, audio_key, text_key) tuple
def _make_stage_a_stream(shuffle=False, take_n=None):
    global STAGE_A_SOURCE, STAGE_A_SOURCE_LABEL
    if STAGE_A_SOURCE == "ted":
        stream = load_dataset("sanchit-gandhi/tedlium-data", split="train", streaming=True)
        STAGE_A_SOURCE_LABEL = "TEDLIUM"
        text_key = ["text", "transcript", "transcription", "sentence"]
        audio_key = "audio"
    if shuffle:
        stream = stream.shuffle(seed=CURRENT_RUN_SEED, buffer_size=STAGE_A_SHUFFLE_BUFFER)
    if take_n is not None:
        stream = stream.take(take_n)
    return stream, audio_key, text_key

# Create DataLoader for Stage-A pretraining data
# input: batch_size, shuffle, take_n, tokenizer, feature_extractor
# process: Calls _make_stage_a_stream, wraps in make_stream_loader
# output: DataLoader yielding batched Stage-A samples (TEDLIUM)
def make_stage_a_loader(batch_size=2, shuffle=False, take_n=None, tokenizer=None, feature_extractor=None):
    stream, audio_key, text_key = _make_stage_a_stream(shuffle=shuffle, take_n=take_n)
    return make_stream_loader(stream, audio_key=audio_key, text_key=text_key, tokenizer=tokenizer, feature_extractor=feature_extractor, batch_size=batch_size)

# Main Execution part
print("\n" + "=" * 80)
print(f"RUNNING SEED {TARGET_SEED}")
print("=" * 80)
seed = TARGET_SEED
set_seed(seed)
CURRENT_RUN_SEED = seed

processor = WhisperProcessor.from_pretrained(model_name, language="en", task="transcribe")
tokenizer = processor.tokenizer
feature_extractor = processor.feature_extractor

model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.max_length = None
if hasattr(model.config, "max_length"):
    model.config.max_length = None

@torch.no_grad()
# Evaluate WER on batches
# input: loader (DataLoader), model (WhisperForConditionalGeneration), device, max_batches
# process: Generate predictions by batch, compute WER
# output: {'wer': float, 'pred_texts': list, 'ref_texts': list}
def eval_wer_from_loader(loader, max_batches=50):
    was_training = model.training
    model.eval()
    preds, refs = [], []
    for i, batch in enumerate(tqdm(loader, desc="Eval"), start=1):
        x = batch["input_features"].to(device, non_blocking=True)
        pred_ids = model.generate(x, **GEN_KW)
        pred_texts = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        preds.extend([norm(t) for t in pred_texts])
        refs.extend([norm(t) for t in batch["texts"]])
        if i >= max_batches:
            break
        if i % 10 == 0:
            cleanup_memory()
    result = wer_metric.compute(predictions=preds, references=refs)
    if was_training:
        model.train()
    return result

@torch.no_grad()
# Compute WER for single batch (helper function for eval_wer_from_loader)
# input: batch dict with input_features/labels/texts, model, device, normalize
# process: Including model inference, generate token IDs, decode predictions, compare with references
# output: (pred_texts, ref_texts) tuple for WER aggregation
def eval_wer_from_batch(batch):
    was_training = model.training
    model.eval()
    x = batch["input_features"].to(device, non_blocking=True)
    pred_ids = model.generate(x, **GEN_KW)
    pred_texts = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    preds = [norm(t) for t in pred_texts]
    refs = [norm(t) for t in batch["texts"]]
    result = wer_metric.compute(predictions=preds, references=refs)
    if was_training:
        model.train()
    return result

# Stage-A cache mechanism: check if a pre-trained checkpoint exists for the current Stage-A configuration (seed, data source, etc.) by hashing these hyperparameters to create a unique cache key. If the cache exists and USE_STAGEA_CACHE is True, load the checkpoint; otherwise, train Stage-A from scratch and save to cache for future runs.
# Pseudo-code:
#   input: seed, model_name, STAGE_A_SOURCE, SPGI_CONFIG, BATCH_SIZE, E22_A_PRETRAIN_STEPS
#   process:
#     1. Create cache_meta dict from all Stage-A hyperparameters
#     2. Hash cache_meta using MD5 - 16-char deterministic cache key
#     3. Construct cache path: BASE_CACHE/stageA_cache/stageA_{key}.pt
#     4. If cache file exists AND USE_STAGEA_CACHE=True: Load pre-trained Stage-A checkpoint (skip 100 training steps)
#     5. Else: Train Stage-A from scratch (100 steps on TEDLIUM)
#   output: model with Stage-A weights (loaded or freshly trained)
# POSSIBLE CASES where cache might miss:
#   1. Path mismatch.
#   2. Key mismatch: Different STAGE_A_SOURCE or seed changes MD5 hash
#   3. Incomplete Export: .pt files not in export, fix: copy to BASE_CACHE

# Stage-A pretrain
cache_meta = {
    "seed": seed,
    "model_name": model_name,
    "stage_a_source": STAGE_A_SOURCE,
    "spgi_config": SPGI_CONFIG,
    "batch_size": BATCH_SIZE,
    "stage_a_steps": E22_A_PRETRAIN_STEPS,
}
stagea_cache_key = hashlib.md5(json.dumps(cache_meta, sort_keys=True).encode("utf-8")).hexdigest()[:16]
stagea_cache_path = os.path.join(STAGEA_CACHE_DIR, f"stageA_{stagea_cache_key}.pt")

if USE_STAGEA_CACHE and os.path.exists(stagea_cache_path):
    print(f"Loading cached Stage-A checkpoint: {stagea_cache_path}")
    cached_stage_a = torch.load(stagea_cache_path, map_location=cpu)
    model.load_state_dict(cached_stage_a["stageA_state"])
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-6)
    a_train_loader = make_stage_a_loader(batch_size=BATCH_SIZE, shuffle=True, tokenizer=tokenizer, feature_extractor=feature_extractor)
    print("Stage-A source:", STAGE_A_SOURCE_LABEL if STAGE_A_SOURCE_LABEL else "SPGISpeech/S")
    model.train()
    pbar_a = tqdm(a_train_loader, desc="Stage-A train")
    for step_a, batch in enumerate(pbar_a, start=1):
        x = batch["input_features"].to(device, non_blocking=True)
        y = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        loss_a = model(input_features=x, labels=y).loss
        loss_a.backward()
        optimizer.step()

        if step_a % 20 == 0:
            pbar_a.set_postfix(loss=float(loss_a.detach().cpu()))
        if step_a >= E22_A_PRETRAIN_STEPS:
            break
    if USE_STAGEA_CACHE:
        stageA_state_to_save = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save({"stageA_state": stageA_state_to_save, "cache_meta": cache_meta}, stagea_cache_path)
        print(f"Saved Stage-A cache: {stagea_cache_path}")

cleanup_memory()
# Baseline eval
a_before_b = eval_wer_from_loader(
    make_e22_loader("val", task_name="A", batch_size=2, take_n=E22_EVAL_N, tokenizer=tokenizer, feature_extractor=feature_extractor),
    max_batches=50,
)
b_before_b = eval_wer_from_loader(
    make_e22_loader("val", task_name="B", batch_size=2, take_n=E22_EVAL_N, tokenizer=tokenizer, feature_extractor=feature_extractor),
    max_batches=50,
)
print(f"After Stage-A | E22-A: {100 * a_before_b:.2f}% | E22-B: {100 * b_before_b:.2f}%")

theta_star = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
# Diagonal Fisher matrix computation
# Pseudo-code:
#   input: model (Stage-A pretrained), Stage-A data (~800 samples)
#   process:
#     1. Initialize diag_fisher: {param_name - zeros_tensor}
#     2. Set model.eval())
#     3. For each batch in Fisher data:
#        a. Compute loss on Stage-A task
#        b. Backward pass to collect gradients
#        c. Accumulate squared gradients: diag_fisher[n] += grad[n]**2
#        d. Clear GPU memory periodically
#     4. Average by num_batches: diag_fisher[n] /= num_batches
#   output: diag_fisher dict with averaged squared gradients per parameter
#
# Used as regularization weights in EWC. Parameters with large squared gradients (important for Stage-A) get higher regularization strength.
# Memory: O(num_params) storage, manageable for Whisper-small (~400M params)
stageA_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

# Compute diagonal Fisher.
print("Computing diagonal Fisher...")
diag_fisher = {n: torch.zeros_like(p) for n, p in model.named_parameters() if p.requires_grad}
a_fisher_loader = make_stage_a_loader(batch_size=BATCH_SIZE, shuffle=True, take_n=E22_A_FISHER_N, tokenizer=tokenizer, feature_extractor=feature_extractor)
model.eval()
num_batches = 0
for batch in tqdm(a_fisher_loader, desc="Diagonal Fisher"):
    x = batch["input_features"].to(device, non_blocking=True)
    y = batch["labels"].to(device, non_blocking=True)
    model.zero_grad(set_to_none=True)
    loss = model(input_features=x, labels=y).loss
    loss.backward()
    for n, p in model.named_parameters():
        if p.grad is not None:
            diag_fisher[n] += p.grad.detach().pow(2)
    num_batches += 1
    if num_batches % 10 == 0:
        cleanup_memory()
for n in diag_fisher:
    diag_fisher[n] /= max(1, num_batches)
print("Diagonal Fisher done.")
cleanup_memory()

# Compute mini-KFAC Fisher
print("Computing mini-KFAC Fisher...")
accum_device = cpu if MINIKFAC_ACCUM_ON_CPU else device
layer_acts = {}
layer_grads = {}
module_param_names = {}
layer_factors = {}
row_sample_calls = [0]

# Memory-efficient subsampling for large activation/gradient matrices
# input: x (tensor), max_rows (max samples to keep)
# process: If x.shape[0] <= max_rows, return full x; else random sample max_rows indices through seed by CURRENT_RUN_SEED
# output: x subsampled to max_rows along first dimension
def _maybe_subsample_rows(x, max_rows):
    if max_rows is None or max_rows <= 0 or x.shape[0] <= max_rows:
        return x
    row_sample_calls[0] += 1
    g = torch.Generator(device=x.device)
    g.manual_seed(CURRENT_RUN_SEED * 1_000_003 + row_sample_calls[0])
    idx = torch.randperm(x.shape[0], generator=g, device=x.device)[:max_rows]
    return x.index_select(0, idx)

# Hook to capture layer input activations for mini-KFAC computation
# input: module (nn.Module), inp (tuple of inputs), out (output)
# process: Extract first input, reshape if needed to, then store in layer_acts[module]
# output: None
def forward_hook(module, inp, out):
    x = inp[0] if isinstance(inp, tuple) else inp
    if x.ndim > 2:
        x = x.reshape(-1, x.shape[-1])
    layer_acts[module] = x.detach().float()

# Hook to capture layer output gradients for mini-KFAC computation
# input: module (nn.Module), grad_in, grad_out (tuple of gradients)
# process: Extract first output gradient, reshape if needed to [batch, features], store in layer_grads[module]
# output: None
def backward_hook(module, grad_in, grad_out):
    g = grad_out[0] if isinstance(grad_out, tuple) else grad_out
    if g.ndim > 2:
        g = g.reshape(-1, g.shape[-1])
    layer_grads[module] = g.detach().float()

# MINI-KFAC (Kronecker-factored approx.) Fisher Computation
# Pseudo-code:
#   input: model (Stage-A), Stage-A data (800 samples), Linear layers
#   process:
#     1. Register hooks on all nn.Linear layers:
#        1.1. forward_hook: captures input activations x [batch, in_features]
#        1.2. backward_hook: captures output gradients g [batch, out_features]
#     2. For each batch:
#        2.1. Forward/backward pass (triggers hooks)
#        2.2. For each layer: accumulate A += x.T@x, G += g.T@g
#           (with subsampling to max 100 rows for memory efficiency)
#     3. Average by num_batches, regularize with 1e-3*I
#     4. Compute layer scales: trace(A)^0.5 * trace(G)^0.5
# ---
# EWC loss computation 
# Pseudo-code:
#   input:
#     model: current model with Stage-B adapted weights
#     fisher: Fisher info
#     theta_star: original Stage-A weights (protected reference)
#     lambda_dict: per-parameter regularization strength
#     fisher_type: 'diagonal' or 'minikfac'
#     lambda_scale: global scaling factor (ramped 0->1 during training)
#   process:
#     Diagonal EWC:
#       penalty += 0.5 * lambda * F[i] * (theta[i] - theta_star[i])^2
#     Mini-KFAC EWC:
#       penalty += 0.5 * lambda * scale * trace(dW^T @ G @ dW @ A)
#       where dW = theta - theta_star
#   output: scalar penalty tensor
# By doing so, we can regularize parameter changes away from Stage-A solution, preventing catastrophic forgetting while allowing Stage-B adaptation.
#     5. Normalize scales (median-based, clipped 1.0-5.0x)
#   output: layer_factors {layer -> {A, G, scale}} dicts
# Some extra notes:
#   Approximate Hessian: H (Kronecker product)
#   A: input covariance (captures activation structure)
#   G: gradient covariance (captures backprop impact)
#   This captures layer-wise curvature without storing full Hessian (that would blow up memory)

hooks = []
linear_modules = [(n, m) for n, m in model.named_modules() if isinstance(m, nn.Linear)]
for layer_idx, (mod_name, mod) in enumerate(linear_modules):
    if MINIKFAC_ONLY_ENCODER and "model.encoder" not in mod_name:
        continue
    if MINIKFAC_LAYER_STRIDE > 1 and (layer_idx % MINIKFAC_LAYER_STRIDE) != 0:
        continue
    if max(mod.in_features, mod.out_features) > MINIKFAC_MAX_FEATURES:
        continue
    w_name = f"{mod_name}.weight"
    b_name = f"{mod_name}.bias" if mod.bias is not None else None
    module_param_names[mod] = (w_name, b_name)
    layer_factors[mod] = {
        "A": torch.zeros(mod.in_features, mod.in_features, device=accum_device),
        "G": torch.zeros(mod.out_features, mod.out_features, device=accum_device),
        "scale": 1.0,
    }
    hooks.append(mod.register_forward_hook(forward_hook))
    if hasattr(mod, "register_full_backward_hook"):
        hooks.append(mod.register_full_backward_hook(backward_hook))
    else:
        hooks.append(mod.register_backward_hook(backward_hook))

a_fisher_loader = make_stage_a_loader(batch_size=BATCH_SIZE, shuffle=True, take_n=MINIKFAC_FISHER_N, tokenizer=tokenizer, feature_extractor=feature_extractor)
model.eval()
num_batches = 0

try:
    for batch in tqdm(a_fisher_loader, desc="mini-KFAC Fisher"):
        x = batch["input_features"].to(device, non_blocking=True)
        y = batch["labels"].to(device, non_blocking=True)
        model.zero_grad(set_to_none=True)
        loss = model(input_features=x, labels=y).loss
        loss.backward()
        for mod, (w_name, b_name) in module_param_names.items():
            if mod not in layer_acts or mod not in layer_grads:
                continue
            act = layer_acts[mod]
            grad = layer_grads[mod]
            if act.numel() == 0 or grad.numel() == 0:
                continue
            act = _maybe_subsample_rows(act, MINIKFAC_MAX_ROWS)
            grad = _maybe_subsample_rows(grad, MINIKFAC_MAX_ROWS)
            if accum_device.type == "cpu":
                act = act.to(cpu)
                grad = grad.to(cpu)
            layer_factors[mod]["A"] += (act.transpose(0, 1) @ act) / max(1, act.shape[0])
            layer_factors[mod]["G"] += (grad.transpose(0, 1) @ grad) / max(1, grad.shape[0])
        layer_acts.clear()
        layer_grads.clear()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        num_batches += 1
finally:
    for h in hooks:
        h.remove()

for mod, stats in layer_factors.items():
    stats["A"] /= max(1, num_batches)
    stats["G"] /= max(1, num_batches)
    stats["A"] += 1e-3 * torch.eye(stats["A"].shape[0], device=stats["A"].device, dtype=stats["A"].dtype)
    stats["G"] += 1e-3 * torch.eye(stats["G"].shape[0], device=stats["G"].device, dtype=stats["G"].dtype)
    a_trace = torch.trace(stats["A"]).item() / max(1, stats["A"].shape[0])
    g_trace = torch.trace(stats["G"]).item() / max(1, stats["G"].shape[0])
    stats["scale"] = float((max(a_trace, 1e-12) * max(g_trace, 1e-12)) ** MINIKFAC_TRACE_EXPONENT)

if MINIKFAC_STRICT_LAYERWISE and len(layer_factors) > 0:
    raw_scales = np.array([stats["scale"] for stats in layer_factors.values()], dtype=np.float64)
    ref = float(np.median(raw_scales)) if raw_scales.size > 0 else 1.0
    ref = max(ref, 1e-12)
    for stats in layer_factors.values():
        ratio = stats["scale"] / ref
        stats["scale"] = float(np.clip(ratio, MINIKFAC_LAYER_SCALE_MIN, MINIKFAC_LAYER_SCALE_MAX))
else:
    for stats in layer_factors.values():
        stats["scale"] = 1.0

print("mini-KFAC Fisher done.")
print(f"mini-KFAC selected layers: {len(layer_factors)}")
cleanup_memory()

# EWC helpers
# Build per-component EWC lambda dictionary for model regularization
# input: base (default=5000), enc (encoder=10000), dec (decoder=2500)
# process: Creates dict mapping module prefixes to lambda weights: encoder/decoder/other
# output: {prefix: lambda_value} dict for use in compute_ewc_loss
def build_lambda_dict(base=LAMBDA_BASE, enc=LAMBDA_ENC, dec=LAMBDA_DEC):
    out = {}
    for n, _ in model.named_parameters():
        if "model.encoder" in n:
            out[n] = enc
        elif "model.decoder" in n:
            out[n] = dec
        else:
            out[n] = base
    return out

# Compute EWC regularization loss with ramp schedule methods
# input: model (current weights), fisher_dict (diagonal/Mini-KFAC), theta_dict (Task-A), lambda_dict
# process: Penalty = 0.5*lambda*F*(theta - theta*)**2, apply min(1.0, step/50) ramp for adaptation
# output: scalar tensor for loss backward pass
def compute_ewc_loss(model, fisher, theta_star, lambda_dict, fisher_type="diagonal", lambda_scale=1.0):
    named_params = dict(model.named_parameters())
    penalty = torch.tensor(0.0, device=device)

    # Handle baseline case (no regularization)
    if fisher is None:
        return penalty
    if fisher_type == "minikfac":
        for mod, stats in fisher.items():
            w_name, b_name = module_param_names[mod]
            layer_scale = float(stats.get("scale", 1.0))
            delta_w = named_params[w_name] - theta_star[w_name]
            G = stats["G"]
            A = stats["A"]
            if G.device != delta_w.device:
                G = G.to(delta_w.device)
            if A.device != delta_w.device:
                A = A.to(delta_w.device)
            weight_penalty = torch.trace(delta_w.t() @ G @ delta_w @ A)
            penalty = penalty + 0.5 * lambda_scale * layer_scale * lambda_dict[w_name] * weight_penalty
            if b_name is not None and b_name in fisher:
                delta_b = named_params[b_name] - theta_star[b_name]
                bias_penalty = torch.sum(fisher[b_name].to(delta_b.device) * delta_b.pow(2))
                penalty = penalty + 0.5 * lambda_scale * lambda_dict[b_name] * bias_penalty
    # diagonal case
    else:
        for n, p in named_params.items():
            if n in fisher:
                diff = p - theta_star[n]
                penalty = penalty + 0.5 * lambda_scale * lambda_dict[n] * torch.sum(fisher[n] * diff.pow(2))

    return penalty

# Compute L2 norm of gradient list for diagnostic tracking
# input: grads (list of gradient tensors, some may be None)
# process: Sum squares of all non-None gradients, take sqrt
# output: Scalar L2 norm value (float)
def grad_l2_norm(grads):
    total = None
    for g in grads:
        if g is None:
            continue
        term = g.detach().pow(2).sum()
        total = term if total is None else (total + term)
    if total is None:
        return 0.0
    return float(torch.sqrt(total + 1e-12).item())

# This is main Stage-B training loop for continual learning experiment runner
# input: exp_name (baseline/ewc/miniKFAC), use_ewc/use_minikfac flags, seed
# process: Load cached Stage-A, compute Fisher, train 120 steps on Task-B with regularization
# output: Save results dict {'wer': {step: val}, 'forgetting': float} to results.json

def run_stage_b_experiment(name, use_ewc, fisher_map=None, fisher_type="diagonal", lambda_dict=None, ckpt_name=None, lambda_scale=1.0):
    model.load_state_dict(stageA_state)
    optimizer_b = torch.optim.AdamW(model.parameters(), lr=STAGEB_LR)
    b_train_loader = make_e22_loader("train", task_name="B", batch_size=BATCH_SIZE, shuffle=True, tokenizer=tokenizer, feature_extractor=feature_extractor)
    total_update_steps = max(1, (TRAIN_STEPS + GRAD_ACCUM_STEPS - 1) // GRAD_ACCUM_STEPS)
    warmup_steps = int(total_update_steps * STAGEB_WARMUP_RATIO)
    scheduler_b = get_linear_schedule_with_warmup(
        optimizer_b,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_update_steps,
    )
    best_ckpt_path = os.path.join(RUN_DIR, ckpt_name or f"best_stageb_{name}.pt")
    if os.path.exists(best_ckpt_path):
        os.remove(best_ckpt_path)
    steps, eval_steps = [], []
    a_hist, b_hist = [], []
    asr_hist, ewc_hist, total_hist = [], [], []
    ewc_ratio_hist = []
    grad_ratio_hist = []
    asr_grad_norm_hist = []
    ewc_grad_norm_hist = []
    train_wer_hist = []
    val_wer_hist = []
    train_penalty_hist = []
    val_penalty_hist = []
    best_b_wer = float("inf")
    best_step = -1
    bad_eval_count = 0
    last_grad_ratio = float("nan")
    last_asr_grad_norm = float("nan")
    last_ewc_grad_norm = float("nan")
    
    model.train()
    optimizer_b.zero_grad(set_to_none=True)
    pbar = tqdm(b_train_loader, desc=f"Stage-B {name}")
    for step, batch in enumerate(pbar, start=1):
        x = batch["input_features"].to(device, non_blocking=True)
        y = batch["labels"].to(device, non_blocking=True)
        asr_loss = model(input_features=x, labels=y).loss
        if use_ewc:
            ramp = min(1.0, step / max(1, EWC_RAMP_STEPS))
            effective_lambda_scale = lambda_scale * ramp
            ewc_loss = compute_ewc_loss(
                model, fisher_map, theta_star, lambda_dict,
                fisher_type=fisher_type, lambda_scale=effective_lambda_scale
            )
        else:
            effective_lambda_scale = 0.0
            ewc_loss = torch.tensor(0.0, device=device)
        grad_ratio = float("nan")
        asr_grad_norm = float("nan")
        ewc_grad_norm = float("nan")
        if use_ewc and (step % GRAD_DIAG_INTERVAL == 0):
            params = [p for p in model.parameters() if p.requires_grad]
            asr_grads = torch.autograd.grad(asr_loss, params, retain_graph=True, allow_unused=True)
            ewc_grads = torch.autograd.grad(ewc_loss, params, retain_graph=True, allow_unused=True)
            asr_grad_norm = grad_l2_norm(asr_grads)
            ewc_grad_norm = grad_l2_norm(ewc_grads)
            grad_ratio = ewc_grad_norm / (asr_grad_norm + 1e-12)
            last_grad_ratio = grad_ratio
            last_asr_grad_norm = asr_grad_norm
            last_ewc_grad_norm = ewc_grad_norm
        else:
            grad_ratio = last_grad_ratio
            asr_grad_norm = last_asr_grad_norm
            ewc_grad_norm = last_ewc_grad_norm

        loss = asr_loss + ewc_loss
        (loss / GRAD_ACCUM_STEPS).backward()
        if (step % GRAD_ACCUM_STEPS == 0) or (step >= TRAIN_STEPS):
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer_b.step()
            scheduler_b.step()
            optimizer_b.zero_grad(set_to_none=True)

        steps.append(step)
        asr_hist.append(float(asr_loss.detach().cpu()))
        ewc_hist.append(float(ewc_loss.detach().cpu()))
        total_hist.append(float(loss.detach().cpu()))
        ratio = float((ewc_loss.detach() / (asr_loss.detach() + 1e-12)).cpu())
        ewc_ratio_hist.append(ratio)
        grad_ratio_hist.append(grad_ratio)
        asr_grad_norm_hist.append(asr_grad_norm)
        ewc_grad_norm_hist.append(ewc_grad_norm)

        if step % 10 == 0:
            pbar.set_postfix(
                asr=asr_hist[-1],
                ewc=ewc_hist[-1],
                total=total_hist[-1],
                ewc_scale=f"{effective_lambda_scale:.2e}" if use_ewc else "0.00e+00",
                ratio=f"{ratio:.2e}",
            )
        if step % EVAL_INTERVAL == 0:
            train_batch_wer = eval_wer_from_batch(batch)
            train_penalty = float(ewc_loss.detach().cpu())
            a_wer = eval_wer_from_loader(
                make_e22_loader("val", task_name="A", batch_size=2, take_n=E22_EVAL_N, tokenizer=tokenizer, feature_extractor=feature_extractor),
                max_batches=EVAL_MAX_BATCHES,
            )
            b_wer = eval_wer_from_loader(
                make_e22_loader("val", task_name="B", batch_size=2, take_n=E22_EVAL_N, tokenizer=tokenizer, feature_extractor=feature_extractor),
                max_batches=EVAL_MAX_BATCHES,
            )
            val_penalty = float(
                compute_ewc_loss(
                    model, fisher_map, theta_star, lambda_dict,
                    fisher_type=fisher_type, lambda_scale=effective_lambda_scale
                ).detach().cpu()
            )
            eval_steps.append(step)
            a_hist.append(a_wer)
            train_wer_hist.append(train_batch_wer)
            val_wer_hist.append(b_wer)
            train_penalty_hist.append(train_penalty)
            val_penalty_hist.append(val_penalty)
            b_hist.append(b_wer)
            print(f"\n[{name}] Seed {seed} Step {step} | E22-A WER: {100 * a_wer:.1f}% | E22-B WER: {100 * b_wer:.1f}%")
            if b_wer < (best_b_wer - EARLY_STOP_MIN_DELTA):
                best_b_wer = b_wer
                best_step = step
                bad_eval_count = 0
                torch.save(
                    {
                        "step": step,
                        "best_b_wer": float(best_b_wer),
                        "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                    },
                    best_ckpt_path,
                )
                print(f"[{name}] Saved best checkpoint @ step {step} (E22-B WER={100 * b_wer:.2f}%)")
            else:
                bad_eval_count += 1

            if bad_eval_count >= EARLY_STOP_PATIENCE_EVALS:
                print(f"[{name}] Early stopping triggered at step {step}.")
                break
            model.train()
        if step >= TRAIN_STEPS:
            break
        if step % 10 == 0:
            cleanup_memory()
    if os.path.exists(best_ckpt_path):
        best_ckpt = torch.load(best_ckpt_path, map_location=device)
        model.load_state_dict(best_ckpt["model_state_dict"])
        best_step = int(best_ckpt["step"])
        best_b_wer = float(best_ckpt["best_b_wer"])
        print(f"[{name}] Loaded best checkpoint from step {best_step} with E22-B WER={100 * best_b_wer:.2f}%")

    final_a = eval_wer_from_loader(
        make_e22_loader("val", task_name="A", batch_size=2, take_n=E22_EVAL_N, tokenizer=tokenizer, feature_extractor=feature_extractor),
        max_batches=EVAL_MAX_BATCHES,
    )
    final_b = eval_wer_from_loader(
        make_e22_loader("val", task_name="B", batch_size=2, take_n=E22_EVAL_N, tokenizer=tokenizer, feature_extractor=feature_extractor),
        max_batches=EVAL_MAX_BATCHES,
    )
    forgetting_a = final_a - a_before_b
    gain_b = b_before_b - final_b
    return {
        "name": name,
        "best_step": best_step,
        "best_b_wer": float(best_b_wer),
        "final_a": float(final_a),
        "final_b": float(final_b),
        "forgetting_a": float(forgetting_a),
        "gain_b": float(gain_b),
        "steps": steps,
        "eval_steps": eval_steps,
        "a_hist": a_hist,
        "b_hist": b_hist,
        "asr_hist": asr_hist,
        "ewc_hist": ewc_hist,
        "total_hist": total_hist,
        "ewc_ratio_hist": ewc_ratio_hist,
        "grad_ratio_hist": grad_ratio_hist,
        "asr_grad_norm_hist": asr_grad_norm_hist,
        "ewc_grad_norm_hist": ewc_grad_norm_hist,
        "train_wer_hist": train_wer_hist,
        "val_wer_hist": val_wer_hist,
        "train_penalty_hist": train_penalty_hist,
        "val_penalty_hist": val_penalty_hist,
    }

# Run comparison for graphs and tables
lambda_dict = build_lambda_dict()
print("\n" + "=" * 80)
print(f"RUNNING EXPERIMENTS FOR SEED {seed}")
print("=" * 80)
# baseline
res_baseline = run_stage_b_experiment(
    name="baseline_no_regularization",
    use_ewc=False,
    fisher_map=None,
    fisher_type="diagonal",
    lambda_dict=lambda_dict,
    ckpt_name=f"best_stageb_baseline.pt",
    lambda_scale=0.0,
)
cleanup_memory()
# ewc
res_ewc = run_stage_b_experiment(
    name="ewc",
    use_ewc=True,
    fisher_map=diag_fisher,
    fisher_type="diagonal",
    lambda_dict=lambda_dict,
    ckpt_name=f"best_stageb_ewc.pt",
    lambda_scale=EWC_LAMBDA_SCALE,
)
cleanup_memory()
# minikfac
res_mini_kfac = run_stage_b_experiment(
    name="miniKFAC_EWC",
    use_ewc=True,
    fisher_map=layer_factors,
    fisher_type="minikfac",
    lambda_dict=lambda_dict,
    ckpt_name=f"best_stageb_mini_kfac_ewc.pt",
    lambda_scale=MINIKFAC_LAMBDA_SCALE,
)
cleanup_memory()

print(f"\n Seed {seed} Stage-B Comparison:")
for r in [res_baseline, res_ewc, res_mini_kfac]:
    print(
        f"{r['name']}: best_B={100 * r['best_b_wer']:.2f}% @ step {r['best_step']} | "
        f"final_A={100 * r['final_a']:.2f}% | final_B={100 * r['final_b']:.2f}% | "
        f"forgetting_A={100 * r['forgetting_a']:+.2f}% | gain_B={100 * r['gain_b']:+.2f}%"
    )
# Save results for this seed, and run next seed as the print line suggestions.
seed_result = {
    "seed": seed,
    "a_before_b": float(a_before_b),
    "b_before_b": float(b_before_b),
    "baseline": res_baseline,
    "ewc": res_ewc,
    "mini_kfac_ewc": res_mini_kfac,
}
result_path = os.path.join(RUN_DIR, f"seed_{seed}_results.json")
with open(result_path, "w") as f:
    json.dump(seed_result, f, indent=2)
print(f"\n Saved seed {seed} results to: {result_path}")
print(f"\nTo run next seed, change TARGET_SEED and re-run this cell.")
print(f"Then use the aggregation script to combine all results.")